In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, collect_list, sort_array, struct
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("NextLabRecommendation") \
    .getOrCreate()

admissions = spark.read.parquet("parquet/admissions")
labevents = spark.read.parquet("parquet/labevents")

In [2]:
labs = labevents.select(
    "subject_id",
    "hadm_id",
    "itemid",
    "charttime"
).dropna()

In [3]:
window = Window.partitionBy("hadm_id").orderBy("charttime")

labs_seq = labs.withColumn(
    "prev_item",
    lag("itemid").over(window)
)

labs_seq = labs_seq.dropna()

In [4]:
transition_counts = labs_seq.groupBy(
    "prev_item", "itemid"
).count()

In [5]:
from pyspark.sql.functions import sum as spark_sum

total_transitions = transition_counts.groupBy(
    "prev_item"
).agg(
    spark_sum("count").alias("total")
)

transition_prob = transition_counts.join(
    total_transitions,
    on="prev_item"
).withColumn(
    "probability",
    col("count") / col("total")
)

In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_rank = Window.partitionBy("prev_item").orderBy(col("probability").desc())

topk = transition_prob.withColumn(
    "rank",
    row_number().over(window_rank)
).filter(col("rank") <= 5)

In [7]:
def recommend_next_test(test_code):

    recs = topk.filter(
        col("prev_item") == test_code
    ).orderBy("probability", ascending=False)

    recs.show()

In [9]:
recommend_next_test(50868)

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it